# L06 · 欧拉公式（Euler's formula） e^{iθ}=cosθ+isinθ — 旋转因子（twiddle factor）是 FFT 的命根子

**学习目标**
1. 理解欧拉公式 `e^{iθ} = cosθ + i·sinθ`——复指数是单位圆上的旋转，实部=cosθ，虚部=sinθ
2. 手动验证 θ=0, π/2, π, 3π/2 等特殊角度，确认实部、虚部与三角函数逐行吻合
3. 实现 `twiddle(k, n, N)` 旋转因子 `e^{-2πikn/N}`，掌握 DFT 中频率索引 k、时间索引 n、长度 N 的含义
4. 确认旋转因子模长恒为 1，相位均匀递减，步长为 `-2π/N`
5. 建立"旋转因子 → DFT 矩阵 → FFT"的直觉链，为 L37–L39 手写 FFT 做准备

**为什么对 Aurora 重要**：DFT 的核心就是旋转因子 `e^{−2πi·k·n/N}`，FFT 全靠它。理解它，L37-L39 重写 FFT 才不虚。

## 本课剧情：把圆上的位置压成一行

单位圆上角度 θ 处的点坐标是 (cosθ, sinθ)。欧拉公式 e^(iθ) = cosθ + i·sinθ 把这两个实数合写成一个复数（complex number），让 DFT 的旋转因子 e^(-2πikn/N) 成为可用一行代码计算的量。本节从手动验证几个角度开始，最后实现 `twiddle(k, n, N)`。

## 1. 手动拼 e^{iθ}，并验证 = np.exp(iθ)

欧拉公式用一个复数 `cosθ + i·sinθ` 代替"x 坐标"和"y 坐标"两个独立实数。这个压缩让 DFT 的旋转因子写成 `e^{−2πi·k·n/N}`——一行乘法，而不是分别调用两个三角函数。

负号来自 DFT 的定义习惯：分析把信号分解到各频率（frequency）用负向旋转，综合（从频率系数重建信号）用正向旋转。`N` 决定把单位圆等分成多少份：N=8 时相邻旋转因子相差 45°（= 2π/8 rad），N=1024 时步长缩到约 0.35°，对应更细的频率分辨率（frequency resolution）。

## 实验入口：角度、坐标和旋转

同一组角度同时打印为 `cos(θ)`、`sin(θ)` 和 `np.exp(1j*θ)`，逐行对齐三列，观察实部与 cos 完全相等、虚部与 sin 完全相等。

In [ ]:
import numpy as np
theta = np.linspace(0, 2*np.pi, 9)
manual = np.cos(theta) + 1j*np.sin(theta)
print('与 np.exp(iθ) 一致:', np.allclose(manual, np.exp(1j*theta)))
print('每个点模长都是1:', np.allclose(np.abs(manual), 1.0))

## 动手观察：复数就是"旋转的位置"

单位圆上角度 θ 的点坐标是 (cosθ, sinθ)，欧拉公式把这两个实数合并写作 e^(iθ)。下面验证几个特殊角度：0、π/2、π、3π/2。

In [ ]:
import numpy as np

angles = np.array([0, np.pi/2, np.pi, 3*np.pi/2])
z = np.exp(1j * angles)

print('角度 =', np.round(angles, 3))
print('实部 cos =', np.round(z.real, 3))
print('虚部 sin =', np.round(z.imag, 3))
print('复数 z =', np.round(z, 3))


## 代码实验：角度、三角函数和复数坐标对齐

打印同一组角度的 `cos(θ)`、`sin(θ)` 和 `np.exp(1j*θ)` 三列，核对实部、虚部与三角函数的数值是否逐行吻合。

In [ ]:
import numpy as np

angles = np.linspace(0, 2*np.pi, 9)
for theta in angles:
    z = np.exp(1j * theta)
    print(f'theta={theta:5.2f} | cos={z.real:6.3f} | sin={z.imag:6.3f} | z={z.real:6.3f}{z.imag:+6.3f}j')


## 2. ✏️ 实现 FFT 旋转因子 `twiddle(k, n, N)`

`W = e^{−2πi·k·n/N}`

**推理路线**：
1. 计算相位（phase，φ）角：`θ = -2π · k · n / N`（k 是频率索引（frequency index，k），n 是采样点索引，N 是 DFT 总点数）
2. 对相位角取复指数：`np.exp(1j * θ)` = `cosθ + i·sinθ`，结果模（magnitude，r）长恒为 1，落在单位圆上
3. 注意负号由 DFT 定义决定，不能省；`k · n / N` 控制本次旋转在整圈中的比例

**参考输入输出**：`twiddle(k=1, n=1, N=8)` = `exp(-2πi/8)` ≈ `0.707 - 0.707j`（位于单位圆第四象限 45° 处）

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


### 写代码前，先把变量表补完整

写 `twiddle` 前明确三件事：
- 输入：`k`（频率索引）、`n`（时间样本索引）、`N`（DFT 总长度）
- 关键步骤：计算相位 `-2π·k·n/N`，对其取复指数 `np.exp(...)`
- 返回：一个复数，模为 1，辐角为 `-2πkn/N`

In [ ]:
def twiddle(k, n, N):
    # ✏️ TODO: 返回 e^{-2πi·k·n/N}
    return None  # ← 改这里


In [ ]:
import numpy as np
assert abs(twiddle(0, 0, 8) - 1) < 1e-12, 'k·n=0 应为 1'
assert abs(abs(twiddle(3, 5, 8)) - 1) < 1e-12, '模长应为 1'
# 相位方向验证：twiddle(1,1,8) = e^{-2πi/8}，虚部必须为负
import numpy as np
_ref = np.exp(-2j * np.pi * 1 * 1 / 8)
assert abs(twiddle(1, 1, 8) - _ref) < 1e-12, '相位方向错误：公式是 e^{-2πikn/N}，负号不能省'
print('✅ 通过：你握住了 FFT 的旋转因子。')

## 3. 画出单位圆

In [ ]:
import numpy as np, matplotlib.pyplot as plt
fine = np.exp(1j*np.linspace(0, 2*np.pi, 200))
plt.figure(figsize=(4,4))
plt.plot(fine.real, fine.imag); plt.gca().set_aspect('equal')
plt.grid(True, alpha=.3); plt.title('e^{iθ} on the unit circle'); plt.show()

**🔗 Aurora 连接**：朴素 DFT（`src/aurora/audio/transforms.py` 中的 `dft()`）对每个频率索引 k，将信号与一列旋转因子 `twiddle(k, 0..N-1, N)` 做内积——内部用 `np.outer(k_vec, n_vec)` 构造完整旋转因子矩阵，与本课的 `twiddle(k, n, N)` 在数学上完全等价。L37-L39 你会把这个 O(N²) 矩阵乘法重写成 O(N log N) 的 FFT——旋转因子的定义不变，变的是计算顺序。

In [ ]:
angles = np.linspace(0, 2*np.pi, 5)
for theta in angles:
    z = np.exp(1j*theta)
    reconstructed = np.cos(theta) + 1j*np.sin(theta)
    print(f'theta={theta:.2f} | exp={z:.2f} | cos+i sin={reconstructed:.2f} | match={np.allclose(z, reconstructed)}')


## 参数实验：观察相位均匀递减

固定 `k=1`，让 `n` 从 0 到 N-1 遍历，打印每个旋转因子的相位（`np.angle`，单位：弧度（radian）），确认相位均匀递减，步长为 `-2π/N`：

```python
N = 8
for n in range(N):
    w = twiddle(k=1, n=n, N=N)
    print(f'n={n} | angle={np.angle(w):.4f} rad ({np.degrees(np.angle(w)):.1f}°)')
# 预期：每行角度差 -45°（= -2π/8 ≈ -0.785 rad）
```

再把 N 改成 4（步长变 -90°）或 1024（步长缩到约 -0.35°），感受 N 对频率分辨率的控制。

In [ ]:
import numpy as np

# 学习目标 #4：确认旋转因子相位均匀递减，步长为 -2π/N
N = 8
for n in range(N):
    w = twiddle(k=1, n=n, N=N)
    print(f'n={n} | angle={np.angle(w):.4f} rad ({np.degrees(np.angle(w)):.1f}°)')

# 断言：相位步长一致性验证
phases = [np.angle(twiddle(1, n, 8)) for n in range(8)]
diffs = np.diff(np.unwrap(phases))
assert np.allclose(diffs, -2 * np.pi / 8), '相位步长不均匀，检查 twiddle 实现'
print(f'\n✅ 相位步长均匀，每步 {-2*np.pi/8:.4f} rad（= -2π/8）')
print('再把 N 改成 4 或 1024，感受 N 对频率分辨率的控制。')


## 本课收束

现在可以用 `np.exp(1j*theta)` 从任意角度算出单位圆上的复数坐标，也可以用 `twiddle(k, n, N)` 算出 DFT 的旋转因子 e^(-2πikn/N)。`twiddle` 直接对应 Aurora 音频核中 `transforms.py` 的 `dft()` 实现逻辑，L37-L39 的 FFT 实现会把它复用。下一课：**L07** 方波谐波叠加——用正弦波叠加近似方波，直观演示傅里叶级数的合成过程，建立「任何周期信号都是正弦波之和」的直觉。

`twiddle` 直接对应 `L37_dft.ipynb` 中 DFT 矩阵的构造——每一行都是一组旋转因子，与 `twiddle(k, n, N)` 数学等价。